# Task 2 - Streaming Application

This notebook implements the streaming pipeline for the AWAS traffic mnonitoring system, which covers Kafka stream ingestion, stream-to-static joins with camera metadata, violation detection (average and instantaneous), and MongoDB sink integration.

Pipeline:
1. Ingesting camera event streams from Kafka through Producers A, B, C which each reads data from camera-events-A, camera-events-B, and camera-events-C respectively.
2. Joining each stream with camera metadata to retrieve speed limit, position, and coordinates
3. Detecting instantaneous violations, vehicles whose recorded speed exceeds the camera's speed limit will be flagged
4. Detecting average speed violations, by joniing entry and exit events then calculating the average speed, we can check if the speed limit has been breached (Road A->B and Road B->C)
5. Pushing all violations to MongoDB, grouped by car plate and violation date

### Environment Setup + Spark


In [1]:
import glob
import os
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date, sin, cos, sqrt, atan2, radians
)
from pyspark.sql.types import *
from datetime import datetime
import time

HOST_IP = "192.168.64.1"
MONGO_URI = "mongodb://mongodb:27017/"
MONGO_DB = "fit3182_a2"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

Debug: SparkSession has been created successfully.


## Watermark Calculation

In [2]:
import pandas as pd

df_a = pd.read_csv(f"{Path('..')}/data/camera_event_A.csv")
df_b = pd.read_csv(f"{Path('..')}/data/camera_event_B.csv")
df_c = pd.read_csv(f"{Path('..')}/data/camera_event_C.csv")
df_a["timestamp"] = pd.to_datetime(df_a["timestamp"])
df_b["timestamp"] = pd.to_datetime(df_b["timestamp"])
df_c["timestamp"] = pd.to_datetime(df_c["timestamp"])

a_timestamp = df_a["timestamp"].cummax()
a_diff = (df_a["timestamp"] - a_timestamp).dt.total_seconds()
a_diff.min() # Minimum time difference to get the latest timestamp for watermarking
camera_a_watermark = round(-a_diff.min() + 0.5) # ensure always round up to the nearest second
print(f"largest gap between an out-of-order event and the maximum timestamp for camera_event_A: {camera_a_watermark} seconds")

b_timestamp = df_b["timestamp"].cummax()
b_diff = (df_b["timestamp"] - b_timestamp).dt.total_seconds()
b_diff.min() # Minimum time difference to get the latest timestamp for watermarking
camera_b_watermark = round(-b_diff.min() + 0.5) # ensure always round up to the nearest second
print(f"largest gap between an out-of-order event and the maximum timestamp for camera_event_B: {camera_b_watermark} seconds")

c_timestamp = df_c["timestamp"].cummax()
c_diff = (df_c["timestamp"] - c_timestamp).dt.total_seconds()
c_diff.min() # Minimum time difference to get the latest timestamp for watermarking
camera_c_watermark = round(-c_diff.min() + 0.5) # ensure always round up to the nearest second
print(f"largest gap between an out-of-order event and the maximum timestamp for camera_event_C: {camera_c_watermark} seconds")

largest gap between an out-of-order event and the maximum timestamp for camera_event_A: 6 seconds
largest gap between an out-of-order event and the maximum timestamp for camera_event_B: 647 seconds
largest gap between an out-of-order event and the maximum timestamp for camera_event_C: 1834 seconds


## Task 2.1.2 Stream Ingestion
Each Kafka topic (camera-events-A/B/C) will be mapped to one producer, its events are then consumed as JSON and parsed against a fixed schema, while also being watermarked to bound the join window.

Event Schema:

| Field | Type | Description |
|---|---|---|
| `event_id` | String | Unique identifier for the camera event |
| `batch_id` | Integer | Producer batch sequence number |
| `car_plate` | String | Vehicle licence plate |
| `camera_id` | Integer | Camera that recorded the event |
| `timestamp` | String | ISO timestamp of the recording |
| `speed_reading` | Double | Recorded speed in km/h |

### Watermarking
Each stream has independent watermarks which have been precalculated based on the data. This preprocessing calculation calculates across the entire csv (data) what the largest gap between an out-of-order event and the maximum timestamp Spark had seen before it arrives. This ensures that no valid event is incorreclty dropped by Spark's late-arrival policy. Since the watermark is tight enough to tolerate the absolute worst-case out-of-order arrival in each stream without being large, which can cause state to accumulate in memory longer than needed if not configured properly.


In [3]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer, watermark_time):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"kafka:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", f"{watermark_time} seconds")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1", camera_a_watermark)
camera_stream_b = read_camera_stream("camera-events-B", "2", camera_b_watermark)
camera_stream_c = read_camera_stream("camera-events-C", "3", camera_c_watermark)

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


## Stream Enrichment
Each stream is joined with the static data of camera.csv to enrich the stream with attributes such as `speed_limit`, `position`, and GPS coordinates (`latitude`, `longitude`) which are needed for violation detection and also distance calculation using Haversine.

This stream-static join is used rather than stream-stream join since camera metadata is fixed and doesn't actually change. So we can simply load it as a DataFrame and join them with our camera events stream.

In [4]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit", "latitude", "longitude")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

# Use pandas for preprocessing since we need row-by-row iteration
camera_pd = camera_df.toPandas().sort_values("camera_id").reset_index(drop=True)
camera_pd["camera_id"] = camera_pd["camera_id"].astype(int)

# Calculate max travel time between adjacent cameras
# position is in km, speed_limit in km/h, result in seconds
camera_times = {}
for i in range(1, len(camera_pd)):
    prev_camera = str(int(camera_pd.iloc[i - 1]["camera_id"]))
    curr_camera = str(int(camera_pd.iloc[i]["camera_id"]))
    distance = camera_pd.iloc[i]["position"] - camera_pd.iloc[i - 1]["position"]
    speed_limit = camera_pd.iloc[i]["speed_limit"]
    camera_times[prev_camera, curr_camera] = round(distance / speed_limit * 3600, 9)

print(f"Camera segment travel times (seconds): {camera_times}")

camera_ids = camera_pd["camera_id"].astype(str).tolist()

# Round down with int for some leniency, since we do checking still and not just based on join condition
max_travel_ab = camera_times[(camera_ids[0], camera_ids[1])]
max_travel_bc = camera_times[(camera_ids[1], camera_ids[2])]

print(f"Max travel time between camera 1 and 2: {max_travel_ab} seconds")
print(f"Max travel time between camera 2 and 3: {max_travel_bc} seconds")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

+---------+--------+-----------+-----------+-----------+
|camera_id|position|speed_limit|   latitude|  longitude|
+---------+--------+-----------+-----------+-----------+
|        1|   152.5|        110|2.157730731|102.6601002|
|        2|   153.5|        110|2.162418757|102.6524549|
|        3|   154.5|         90|2.167352891|102.6449144|
+---------+--------+-----------+-----------+-----------+

Debug: Camera loaded: 3 cameras.
Camera segment travel times (seconds): {('1', '2'): 32.727272727, ('2', '3'): 40.0}
Max travel time between camera 1 and 2: 32.727272727 seconds
Max travel time between camera 2 and 3: 40.0 seconds


## Task 2.1.2 — Average Speed Violation Detection: Segment Joins

Average speed violations are detected by joining entry and exit events for the same vehicle across different segments of the road. Since the road and camera placement is strictly in the order A -> B -> C, two segment joins are performed: A->B and B->C.

### Join Strategy
Segment joins use physical time-ordering rather than `batch_id`, since `batch_id` is a producer-side sequence number and is not synchronised across producers. A higher `batch_id` in Producer B does not necessarily guarantee a later `event_time` than Producer A, so using `batch_id` as a join key would potentially drop valid pairs.

Instead, events are matched on `car_plate` with two time-ordering constraints:
1. `exit.event_time > entry.event_time` — ensures the vehicle passes the exit camera **after** the entry camera, since a vehicle cannot travel in reverse.
2. `exit.event_time <= entry.event_time + max_travel_ab/bc` — only retains pairs where the travel time is within the maximum time a vehicle travelling at exactly the speed limit would take. This naturally excludes non-violating pairs (vehicles travelling under the speed limit arrive too late to be within the window) while capturing all violating pairs (speeding vehicles arrive within the window by definition).

`max_travel_ab` and `max_travel_bc` are derived directly from the camera metadata (segment distance and speed limit), ensuring the join window is data-driven rather than an arbitrary constant.

### Dropped Pairs
When no matching exit event arrives for a given entry within the join window, the entry record is eventually evicted from state by the watermark and logged through the logger function. Each stream has its own independently calculated watermark duration based on the maximum observed out-of-order lateness in its respective CSV. Spark computes a global watermark as the **minimum watermark threshold** across all joined streams, taking the stream whose threshold is furthest behind in time (the slowest stream). This means the slowest stream protects all other streams, ensuring no valid pairs are dropped due to one stream advancing faster than another. An unmatched entry event is dropped once its `event_time` falls below the global watermark threshold, at which point Spark considers it impossible for a valid matching exit event to ever arrive.

In [5]:
# A→B segment join (camera 1 to camera 2)
segment_ab = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr(f"""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval {max_travel_ab} seconds
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
        
    )
)

# B→C segment join (camera 2 to camera 3)
segment_bc = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr(f"""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval {max_travel_bc} seconds
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
    )
)

def log_batch_with_drops(name):
    """Log batch details and flag empty batches as dropped pairs."""
    def logger(batch_df, batch_id):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        row_count = batch_df.count()
        if row_count == 0:
            print(f"[{now}] [{name}] Batch {batch_id}: NO matching pairs — records dropped/expired by watermark.")
        else:
            print(f"[{now}] [{name}] Batch {batch_id}: {row_count} matched pair(s).")
            batch_df.show(truncate=False)
    return logger

segment_ab_query = (
    segment_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch_with_drops("Segment A→B"))
    .start()
)

segment_bc_query = (
    segment_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch_with_drops("Segment B→C"))
    .start()
)

print("Debug: Segment joins have been defined for A→B and B→C.")

Debug: Segment joins have been defined for A→B and B→C.


## Task 2.1.3 — MongoDB Sink

Violations are persisted to the `violations` collection in MongoDB using `foreachBatch` with pymongo `bulk_write` and `UpdateOne` upserts.

### Daily Merging (Task 2.1.4)

Multiple violations for the same vehicle on the same day are merged into a single document. The upsert match key is `(car_plate, date)`, meaning one document per car per day. Each new violation is added to a `violations` array within that document via `$addToSet`, so duplicates are avoided.

### Retry Handling

Write failures are retried up to **3 times** with a **2-second delay** between attempts. If all retries are exhausted, the batch is logged as dropped rather than crashing the stream.

### Bulk Writes

All operations within a micro-batch are collected into a single `bulk_write` call with `ordered=False`, which maximises write throughput by allowing MongoDB to execute operations in parallel and not halting on a single failure.

### Indexes

The `violations` collection uses a compound index on `(car_plate, date)` which directly matches the upsert filter key, ensuring O(log n) lookups rather than full collection scans on every write. A secondary index on `date` alone supports time-range queries used in visualisation. See `mongo_setup.py` for index creation.

In [6]:

def mongo_sink(name):
    def write_violations_to_mongo(batch_df, batch_id):
        rows = batch_df.collect()
        
        if not rows:
            print(f"Batch {batch_id} is empty, skipping MongoDB write. resolving: {name}")
            return

        operations = []
        for row in rows:
            doc = row.asDict()

            # Build the sub-document to push into the violations array
            if doc["violation_type"] == "instantaneous":
                violation_entry = {
                    "type":      "instant",
                    "camera_id": doc["camera_id"],
                    "speed":     doc["speed_recorded"],
                }
            else:
                violation_entry = {
                    "type":         "average",
                    "start_camera": doc["start_camera_id"],
                    "end_camera":   doc["end_camera_id"],
                    "avg_speed":    doc["average_speed"],
                }

            operations.append(
                UpdateOne(
                    {
                        "car_plate": doc["car_plate"],
                        "date": datetime.combine(doc["violation_date"], datetime.min.time()),
                    },
                    {
                        "$addToSet": {"violations": {"$each": [violation_entry]}},
                    },
                    upsert=True
                )
            )
        
        MAX_RETRIES = 3
        RETRY_DELAY = 2  # seconds
        
        client = MongoClient(MONGO_URI)
        for attempt in range(1, MAX_RETRIES + 1): # HD Requirement
            try:
                collection = client[MONGO_DB]["violations"]
                result = collection.bulk_write(operations, ordered=False)
                print(
                    f"[Batch {batch_id}] [{name}]: {len(operations)} op(s) — "
                    f"upserted: {result.upserted_count}, modified: {result.modified_count}"
                )
                break  # success, exit retry loop
            except Exception as exc:
                print(f"[Batch {batch_id}] [{name}] Attempt {attempt}/{MAX_RETRIES} failed: {exc}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"[Batch {batch_id}] [{name}] All retries exhausted, batch dropped.")
            finally:
                client.close()
    return write_violations_to_mongo

print("Debug: MongoDB sink function defined.")


Debug: MongoDB sink function defined.


## Task 2.1.4 — Instantaneous Speed Violation Detection

A vehicle is flagged for an instantaneous violation when its `speed_reading` at the recording camera exceeds that camera's `speed_limit`. This check is applied independently to each of the three streams.

Each violation record retains `event_id` for traceability, and `violation_date` (derived from `event_time`) to support the daily merging logic in MongoDB.

In [7]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "event_id",
            "car_plate",
            "batch_id",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        # Append JSON Lines to a single file so batches accumulate.
        file_path = base_path
        if not file_path.endswith(".json"):
            file_path = f"{base_path}/results.json"
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        rows = batch_df.toJSON().collect()
        if not rows:
            return
        with open(file_path, "a", encoding="utf-8") as f:
            for row in rows:
                f.write(row + "\n")
    return _writer

camera_a_instant_query = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
    .start()
)

camera_b_instant_query = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
    .start()
)

camera_c_instant_query = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
    .start()
)

print("Debug: Instantaneous violations have been extracted and combined.")


Debug: Instantaneous violations have been extracted and combined.


## Task 2.1.4 — Average Speed Violation Detection: Computation

For each matched entry/exit pair from the segment joins, the average speed across the segment is computed as:
```
average_speed (km/h) = distance_km / travel_time_hours
```

**Distance** can be calculated by using either the Haversine formula using the given `latitude` and `longitude` or using the `position` column. Based on the teaching team feedback, it is advised for now that we use the `position` column to calculate our distance between cameras so that it is a round number.

**Travel time** is derived by casting both `entry_time` and `exit_time` to doubles, taking their difference, and dividing by 3600 to convert to hours.

A pair is flagged as a violation only when `average_speed > speed_limit` of the 
**exit camera**, consistent with the AWAS point-to-point enforcement model. The `violation_date` is derived from the `exit_time`, since the exit event is when the violation is confirmed.

In [8]:


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1_rad = radians(col(lat1))
    lon1_rad = radians(col(lon1))
    lat2_rad = radians(col(lat2))
    lon2_rad = radians(col(lon2))

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        sin(delta_lat / 2) * sin(delta_lat / 2)
        + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2) * sin(delta_lon / 2)
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance_km = R * c
    return distance_km

def compute_avg_speed(joined_segments):
    return (
        joined_segments
        .withColumn(
            "distance_km",
            col("exit_position") - col("entry_position")
        )
        .withColumn(
            "travel_time_hours",
            (col("exit_time").cast("double") - col("entry_time").cast("double")) / 3600
                    )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time",
            "entry_batch_id",
            "exit_batch_id"
    )
)


average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

ab_avg_violations_query = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_ab"))
    .start()
)

bc_avg_violations_query = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_bc"))
    .start()
)

# avg_vio_ab_query = (
#     average_violations_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations A→B"))
#     .start()
# )

# avg_vio_bc_query = (
#     average_violations_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations B→C"))
#     .start()
# )


print("Average speed violation detection logic defined.")

Average speed violation detection logic defined.


## Task 2.2.1 — Folium Live Map

This section aggregates per-camera stats from the live streams and renders a Folium map with three markers. Each marker shows the current violation count and the average speed of all cars recorded at that camera so far.

References:
- https://python-visualization.github.io/folium/quickstart.html
- https://python-visualization.github.io/folium/modules.html#module-folium.folium

Re-run the rendering cell to refresh the values during the simulation.

In [9]:
from pyspark.sql.functions import sum as spark_sum, count as spark_count
from pathlib import Path
import json

CAMERA_STATS_PATH = Path('..') / 'outputs' / 'camera_stats.json'

if not globals().get("CAMERA_STATS_CLEARED", False):
    if CAMERA_STATS_PATH.exists():
        CAMERA_STATS_PATH.unlink()
    CAMERA_STATS_CLEARED = True

def load_camera_stats():
    if CAMERA_STATS_PATH.exists():
        with open(CAMERA_STATS_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {
        "car_count": {},
        "speed_sum": {},
        "violations_instant": {},
        "violations_average": {},
    }

def save_camera_stats(stats):
    CAMERA_STATS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(CAMERA_STATS_PATH, 'w', encoding='utf-8') as f:
        json.dump(stats, f)

def update_speed_stats(batch_df, batch_id):
    agg_df = batch_df.groupBy('camera_id').agg(
        spark_sum('speed_reading').alias('speed_sum'),
        spark_count('*').alias('car_count')
    )
    rows = agg_df.collect()
    if not rows:
        return
    stats = load_camera_stats()
    for row in rows:
        camera_id = str(int(row['camera_id']))
        stats['speed_sum'][camera_id] = stats['speed_sum'].get(camera_id, 0.0) + float(row['speed_sum'])
        stats['car_count'][camera_id] = stats['car_count'].get(camera_id, 0) + int(row['car_count'])
    save_camera_stats(stats)

def update_violation_stats(batch_df, batch_id, key):
    rows = batch_df.groupBy('camera_id').count().collect()
    if not rows:
        return
    stats = load_camera_stats()
    for row in rows:
        camera_id = str(int(row['camera_id']))
        stats[key][camera_id] = stats[key].get(camera_id, 0) + int(row['count'])
    save_camera_stats(stats)

def update_instant_violation_stats(batch_df, batch_id):
    update_violation_stats(batch_df, batch_id, 'violations_instant')

def update_average_violation_stats(batch_df, batch_id):
    update_violation_stats(batch_df, batch_id, 'violations_average')

camera_speed_stream = (
    joined_stream_a.select('camera_id', 'speed_reading')
    .unionByName(joined_stream_b.select('camera_id', 'speed_reading'))
    .unionByName(joined_stream_c.select('camera_id', 'speed_reading'))
)

instant_violation_stream = (
    camera_a_instant_violations.select(col('camera_id'))
    .unionByName(camera_b_instant_violations.select(col('camera_id')))
    .unionByName(camera_c_instant_violations.select(col('camera_id')))
)

average_violation_stream = (
    average_violations_ab.select(col('end_camera_id').alias('camera_id'))
    .unionByName(average_violations_bc.select(col('end_camera_id').alias('camera_id')))
)

print("Debug: Camera stats helpers and stream aggregates defined.")


Debug: Camera stats helpers and stream aggregates defined.


## Starting All Streaming Queries

Each violation type and camera combination is wired to a separate `writeStream` query targeting the MongoDB sink. Running them as independent queries allows Spark to manage their trigger schedules and checkpoints separately.

| Query | Source | Violation Type |
|---|---|---|
| `camera_a_query_mongo` | Stream A | Instantaneous |
| `camera_b_query_mongo` | Stream B | Instantaneous |
| `camera_c_query_mongo` | Stream C | Instantaneous |
| `ab_avg_query_mongo` | Segment A→B | Average speed |
| `bc_avg_query_mongo` | Segment B→C | Average speed |
| `camera_speed_query` | Streams A/B/C | Average speed stats |
| `violation_stats_query` | Instant/avg violations | Violation counts |

In [10]:
camera_a_query_mongo = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_a_instant"))
    .start()
    )

camera_b_query_mongo = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_b_instant"))
    .start()
    )

camera_c_query_mongo = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_c_instant"))
    .start()
    )

ab_avg_query_mongo = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_ab"))
    .start()
    )

bc_avg_query_mongo = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_bc"))
    .start()
    )

camera_speed_query = (
    camera_speed_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(update_speed_stats)
    .start()
    )

instant_violation_stats_query = (
    instant_violation_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(update_instant_violation_stats)
    .start()
    )

average_violation_stats_query = (
    average_violation_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(update_average_violation_stats)
    .start()
    )

print("Debug: MongoDB streaming queries have been started for all violation types.")

Debug: MongoDB streaming queries have been started for all violation types.
Batch 0 is empty, skipping MongoDB write. resolving: camera_a_instant
Batch 0 is empty, skipping MongoDB write. resolving: camera_c_instant
[Batch 0] [camera_b_instant]: 2 op(s) — upserted: 0, modified: 0
[Batch 1] [camera_c_instant]: 7 op(s) — upserted: 0, modified: 0
[Batch 1] [camera_a_instant]: 47 op(s) — upserted: 0, modified: 0
[Batch 1] [camera_b_instant]: 2 op(s) — upserted: 0, modified: 0
[Batch 2] [camera_c_instant]: 1 op(s) — upserted: 0, modified: 0
[Batch 2] [camera_a_instant]: 51 op(s) — upserted: 0, modified: 0
[2026-05-25 11:57:59] [Segment B→C] Batch 0: NO matching pairs — records dropped/expired by watermark.
[2026-05-25 11:57:59] [Segment A→B] Batch 0: NO matching pairs — records dropped/expired by watermark.
[Batch 2] [camera_b_instant]: 6 op(s) — upserted: 0, modified: 0
[Batch 3] [camera_c_instant]: 9 op(s) — upserted: 0, modified: 0
Batch 0 is empty, skipping MongoDB write. resolving: avg

Batch 7 is empty, skipping MongoDB write. resolving: avg_bc
[2026-05-25 12:00:01] [Segment B→C] Batch 7: NO matching pairs — records dropped/expired by watermark.
[2026-05-25 12:00:02] [Segment A→B] Batch 7: NO matching pairs — records dropped/expired by watermark.
Batch 8 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 13] [camera_c_instant]: 13 op(s) — upserted: 0, modified: 0
[Batch 13] [camera_a_instant]: 106 op(s) — upserted: 0, modified: 0
[Batch 12] [camera_b_instant]: 11 op(s) — upserted: 0, modified: 0
Batch 8 is empty, skipping MongoDB write. resolving: avg_bc
[2026-05-25 12:00:10] [Segment A→B] Batch 8: NO matching pairs — records dropped/expired by watermark.
[2026-05-25 12:00:10] [Segment B→C] Batch 8: NO matching pairs — records dropped/expired by watermark.
[Batch 14] [camera_a_instant]: 85 op(s) — upserted: 0, modified: 0
[Batch 14] [camera_c_instant]: 10 op(s) — upserted: 0, modified: 0
[Batch 13] [camera_b_instant]: 9 op(s) — upserted: 0, modified: 0
Batch 

[Batch 30] [camera_c_instant]: 10 op(s) — upserted: 0, modified: 0
[2026-05-25 12:02:52] [Segment B→C] Batch 21: NO matching pairs — records dropped/expired by watermark.
[2026-05-25 12:02:56] [Segment A→B] Batch 22: NO matching pairs — records dropped/expired by watermark.
[Batch 30] [camera_b_instant]: 7 op(s) — upserted: 0, modified: 0
Batch 22 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 31] [camera_a_instant]: 104 op(s) — upserted: 0, modified: 0
[Batch 31] [camera_c_instant]: 6 op(s) — upserted: 0, modified: 0
Batch 23 is empty, skipping MongoDB write. resolving: avg_ab
[2026-05-25 12:03:04] [Segment A→B] Batch 23: NO matching pairs — records dropped/expired by watermark.
[Batch 32] [camera_a_instant]: 116 op(s) — upserted: 0, modified: 0
[Batch 31] [camera_b_instant]: 16 op(s) — upserted: 0, modified: 0
[2026-05-25 12:03:04] [Segment B→C] Batch 22: NO matching pairs — records dropped/expired by watermark.
[Batch 32] [camera_c_instant]: 13 op(s) — upserted: 0, modif

[Batch 49] [camera_a_instant]: 68 op(s) — upserted: 0, modified: 0
[Batch 48] [camera_b_instant]: 4 op(s) — upserted: 0, modified: 0
Batch 36 is empty, skipping MongoDB write. resolving: avg_ab
Batch 35 is empty, skipping MongoDB write. resolving: avg_bc
[2026-05-25 12:05:30] [Segment A→B] Batch 36: NO matching pairs — records dropped/expired by watermark.
[2026-05-25 12:05:34] [Segment B→C] Batch 35: NO matching pairs — records dropped/expired by watermark.
[Batch 50] [camera_c_instant]: 4 op(s) — upserted: 0, modified: 0
[Batch 49] [camera_b_instant]: 3 op(s) — upserted: 0, modified: 0
[Batch 50] [camera_a_instant]: 72 op(s) — upserted: 0, modified: 0
Batch 37 is empty, skipping MongoDB write. resolving: avg_ab
Batch 36 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 50] [camera_b_instant]: 7 op(s) — upserted: 0, modified: 0
[Batch 51] [camera_c_instant]: 12 op(s) — upserted: 0, modified: 0
[2026-05-25 12:05:45] [Segment A→B] Batch 37: NO matching pairs — records dropped/e

[Batch 67] [camera_a_instant]: 105 op(s) — upserted: 0, modified: 0
Batch 49 is empty, skipping MongoDB write. resolving: avg_bc
[Batch 68] [camera_c_instant]: 5 op(s) — upserted: 0, modified: 0
[Batch 66] [camera_b_instant]: 14 op(s) — upserted: 0, modified: 0
Batch 50 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 68] [camera_a_instant]: 131 op(s) — upserted: 0, modified: 0
[Batch 69] [camera_c_instant]: 14 op(s) — upserted: 0, modified: 0
[2026-05-25 12:08:09] [Segment B→C] Batch 49: NO matching pairs — records dropped/expired by watermark.
[Batch 67] [camera_b_instant]: 10 op(s) — upserted: 0, modified: 0
[2026-05-25 12:08:12] [Segment A→B] Batch 50: NO matching pairs — records dropped/expired by watermark.
Batch 50 is empty, skipping MongoDB write. resolving: avg_bc
Batch 51 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 70] [camera_c_instant]: 9 op(s) — upserted: 0, modified: 0
[Batch 69] [camera_a_instant]: 75 op(s) — upserted: 0, modified: 0
[Batch 68] [

...

[Batch 99] [camera_a_instant]: 131 op(s) — upserted: 0, modified: 0
[Batch 74] [avg_bc]: 14 op(s) — upserted: 0, modified: 0
[2026-05-25 12:12:48] [Segment A→B] Batch 74: NO matching pairs — records dropped/expired by watermark.
[Batch 100] [camera_b_instant]: 14 op(s) — upserted: 0, modified: 0
[Batch 101] [camera_c_instant]: 12 op(s) — upserted: 0, modified: 0
Batch 75 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 100] [camera_a_instant]: 156 op(s) — upserted: 0, modified: 0
[2026-05-25 12:12:55] [Segment B→C] Batch 63: 20 matched pair(s).
[Batch 101] [camera_b_instant]: 14 op(s) — upserted: 0, modified: 0
[Batch 75] [avg_bc]: 10 op(s) — upserted: 0, modified: 0
[Batch 102] [camera_c_instant]: 10 op(s) — upserted: 0, modified: 0
[2026-05-25 12:12:59] [Segment A→B] Batch 75: NO matching pairs — records dropped/expired by watermark.
Batch 76 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 101] [camera_a_instant]: 135 op(s) — upserted: 0, modified: 0
[Batch 102] 

[Batch 106] [camera_c_instant]: 12 op(s) — upserted: 0, modified: 0
[Batch 106] [camera_b_instant]: 15 op(s) — upserted: 0, modified: 0
[Batch 105] [camera_a_instant]: 99 op(s) — upserted: 0, modified: 0
Batch 79 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 79] [avg_bc]: 10 op(s) — upserted: 0, modified: 0
[2026-05-25 12:13:43] [Segment A→B] Batch 79: NO matching pairs — records dropped/expired by watermark.
[Batch 107] [camera_c_instant]: 5 op(s) — upserted: 0, modified: 0
[Batch 107] [camera_b_instant]: 14 op(s) — upserted: 0, modified: 0
[Batch 106] [camera_a_instant]: 108 op(s) — upserted: 0, modified: 0
[2026-05-25 12:13:50] [Segment B→C] Batch 65: 22 matched pair(s).
Batch 80 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 80] [avg_bc]: 8 op(s) — upserted: 0, modified: 0
[Batch 108] [camera_c_instant]: 12 op(s) — upserted: 0, modified: 0
[Batch 108] [camera_b_instant]: 7 op(s) — upserted: 0, modified: 0
[Batch 107] [camera_a_instant]: 115 op(s) — upserted

+---------+---------------+-------------+--------------+-------------+--------------------------+--------------------------+--------------+-------------+-----------+------+--------------+---------------+-------------+--------------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time                |exit_time                 |entry_position|exit_position|speed_limit|source|entry_latitude|entry_longitude|exit_latitude|exit_longitude|
+---------+---------------+-------------+--------------+-------------+--------------------------+--------------------------+--------------+-------------+-----------+------+--------------+---------------+-------------+--------------+
|WO 684   |2              |3            |1819          |2554         |2024-01-02 20:21:57.538364|2024-01-02 20:22:26.466356|153.5         |154.5        |90         |3     |2.162418757   |102.6524549    |2.167352891  |102.6449144   |
|UHG 736  |2              |3            |1836          |2556        

[Batch 115] [camera_b_instant]: 5 op(s) — upserted: 0, modified: 0
[Batch 114] [camera_a_instant]: 99 op(s) — upserted: 0, modified: 0
[Batch 115] [camera_c_instant]: 10 op(s) — upserted: 0, modified: 0
[Batch 86] [avg_bc]: 8 op(s) — upserted: 0, modified: 0
[2026-05-25 12:15:07] [Segment A→B] Batch 86: NO matching pairs — records dropped/expired by watermark.
Batch 86 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 116] [camera_b_instant]: 5 op(s) — upserted: 0, modified: 0
[Batch 115] [camera_a_instant]: 133 op(s) — upserted: 0, modified: 0
[2026-05-25 12:15:14] [Segment B→C] Batch 68: 21 matched pair(s).
[Batch 116] [camera_c_instant]: 13 op(s) — upserted: 0, modified: 0
[2026-05-25 12:15:14] [Segment A→B] Batch 87: NO matching pairs — records dropped/expired by watermark.
[Batch 87] [avg_bc]: 13 op(s) — upserted: 0, modified: 0
Batch 87 is empty, skipping MongoDB write. resolving: avg_ab
[Batch 117] [camera_b_instant]: 15 op(s) — upserted: 0, modified: 0
[Batch 117] [cam

[Batch 120] [camera_b_instant]: 19 op(s) — upserted: 0, modified: 0
[Batch 119] [camera_a_instant]: 167 op(s) — upserted: 0, modified: 0
[2026-05-25 12:16:02] [Segment A→B] Batch 90: NO matching pairs — records dropped/expired by watermark.
[Batch 120] [camera_c_instant]: 14 op(s) — upserted: 0, modified: 0
[Batch 90] [avg_bc]: 11 op(s) — upserted: 0, modified: 0
[Batch 121] [camera_b_instant]: 9 op(s) — upserted: 0, modified: 0
Batch 90 is empty, skipping MongoDB write. resolving: avg_ab
[2026-05-25 12:16:11] [Segment B→C] Batch 70: 32 matched pair(s).
[Batch 120] [camera_a_instant]: 123 op(s) — upserted: 0, modified: 0
[Batch 121] [camera_c_instant]: 16 op(s) — upserted: 0, modified: 0
[2026-05-25 12:16:24] [Segment A→B] Batch 91: NO matching pairs — records dropped/expired by watermark.
[Batch 122] [camera_b_instant]: 7 op(s) — upserted: 0, modified: 0
[Batch 91] [avg_bc]: 13 op(s) — upserted: 0, modified: 0
Batch 91 is empty, skipping MongoDB write. resolving: avg_ab
+---------+---